# PourCastAI — Step 6: Bronze + static reference -> Silver

Builds conformed Silver tables, same grain/keys as your original schema.sql:
  dim_store, dim_item, dim_vendor, dim_carrier   (dimensions)
  fact_sales, fact_shipments                      (facts)

Sources:
  dim_store      <- bronze.hubspot_companies (+ store_coords.csv for lat/long)
  fact_shipments <- bronze.hubspot_deals
  everything else <- static CSVs uploaded to the landing Volume (Step 6 export)

In [0]:
CATALOG = "pourcastai"
LANDING = f"/Volumes/{CATALOG}/bronze/landing"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")

DataFrame[]

### dim_store — from HubSpot Companies, demo row filtered out, coords joined in

In [0]:
from pyspark.sql.functions import col

companies = spark.table(f"{CATALOG}.bronze.hubspot_companies") \
    .filter(col("store_number").isNotNull()) \
    .select(
        col("store_number").cast("int"),
        col("name").alias("store_name"),
        col("address"),
        col("city"),
        col("zip").alias("zip_code"),
    )

coords = spark.read.csv(f"{LANDING}/store_coords.csv", header=True, inferSchema=True) \
    .select(col("store_number").cast("int"), "latitude", "longitude",
            "population", col("rural_flag").cast("int"), "county")

dim_store = companies.join(coords, on="store_number", how="left")

dim_store.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.silver.dim_store")
print(f"silver.dim_store: {dim_store.count()} rows")
display(dim_store)

silver.dim_store: 20 rows


store_number,store_name,address,city,zip_code,latitude,longitude,population,rural_flag,county
2614,Hy-Vee #3 Food and Drugstore,1823 E Kimberly Rd,Davenport,52807,41.556781,-90.548919,175000,0,SCOTT
3814,Costco Wholesale #788,7205 Mills Civic Pkwy,West Des Moines,50266,41.561342,-93.806489,100000,0,Dallas
2629,Hy-Vee Food Store #2 / Council Bluffs,1745 Madison Ave,Council Bluffs,51503,41.242732,-95.825137,null,0,POTTAWATTA
3354,Sam's Club 8238 / Davenport,3845 Elmore Ave.,Davenport,52807,41.559731,-90.527081,175000,0,SCOTT
3773,Benz Distributing,501 7th Ave SE,Cedar Rapids,52401,41.975787,-91.659795,230000,0,LINN
3524,Sam's Club 6568 / Ames,305 Airport Rd,Ames,50010,42.001123,-93.61365,100000,0,STORY
3385,Sam's Club 8162 / Cedar Rapids,2605 Blairs Ferry Rd NE,Cedar Rapids,52402,42.031819,-91.67969,230000,0,LINN
2593,Hy-Vee Food Store / Carroll,905 US Highway 30 West,Carroll,51401,42.064155,-94.853591,null,0,CARROLL
2633,Hy-Vee #3 / BDI / Des Moines,3221 SE 14th St,Des Moines,50320,41.554101,-93.596754,500000,0,POLK
3820,"Charlie's Wine and Spirits,",507 W 19th St,Sioux City,51103,42.510535,-96.420193,105000,0,WOODBURY


### dim_item, dim_vendor, dim_carrier — straight loads from the static CSVs

In [0]:
for table in ["dim_item", "dim_vendor", "dim_carrier"]:
    df = spark.read.csv(f"{LANDING}/{table}.csv", header=True, inferSchema=True)
    df.write.mode("overwrite").option("overwriteSchema", "true") \
        .saveAsTable(f"{CATALOG}.silver.{table}")
    print(f"silver.{table}: {df.count()} rows")

silver.dim_item: 25 rows
silver.dim_vendor: 9 rows
silver.dim_carrier: 3 rows


### fact_inventory_snapshot — straight load, needed for gold_inventory_health

In [0]:
fact_inv = spark.read.csv(f"{LANDING}/fact_inventory_snapshot.csv", header=True, inferSchema=True)
fact_inv.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.silver.fact_inventory_snapshot")
print(f"silver.fact_inventory_snapshot: {fact_inv.count()} rows")

silver.fact_inventory_snapshot: 123342 rows


### fact_sales — straight load, this is your REAL historical data, untouched

In [0]:
fact_sales = spark.read.csv(f"{LANDING}/fact_sales.csv", header=True, inferSchema=True)
fact_sales.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.silver.fact_sales")
print(f"silver.fact_sales: {fact_sales.count()} rows")

silver.fact_sales: 9604 rows


### fact_shipments — from HubSpot Deals, vendor_number derived via dim_item join,
### origin/dest coordinates derived (Ankeny is fixed, dest comes from dim_store)

In [0]:
ANKENY_LAT, ANKENY_LONG = 41.699, -93.558

deals = spark.table(f"{CATALOG}.bronze.hubspot_deals") \
    .filter(col("store_number").isNotNull()) \
    .select(
        col("shipment_id").cast("int"),
        col("order_date").cast("date"),
        col("closedate").cast("date").alias("promised_eta"),
        col("store_number").cast("int"),
        col("item_number").cast("int"),
        col("carrier_id").cast("int"),
        col("quantity_bottles").cast("int"),
    )

dim_item_silver = spark.table(f"{CATALOG}.silver.dim_item").select("item_number", "vendor_number")
dim_store_silver = spark.table(f"{CATALOG}.silver.dim_store") \
    .select(col("store_number"), col("latitude").alias("dest_lat"), col("longitude").alias("dest_long"))

from pyspark.sql.functions import lit

fact_shipments = deals \
    .join(dim_item_silver, on="item_number", how="left") \
    .join(dim_store_silver, on="store_number", how="left") \
    .withColumn("origin_lat", lit(ANKENY_LAT)) \
    .withColumn("origin_long", lit(ANKENY_LONG))

fact_shipments.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.silver.fact_shipments")
print(f"silver.fact_shipments: {fact_shipments.count()} rows")
display(fact_shipments)

silver.fact_shipments: 8 rows


store_number,item_number,shipment_id,order_date,promised_eta,carrier_id,quantity_bottles,vendor_number,dest_lat,dest_long,origin_lat,origin_long
2633,11297,323,2026-08-10,2026-08-20,2,114,260,41.554101,-93.596754,41.699,-93.558
3385,11777,1588,2026-08-11,2026-08-19,3,23,115,42.031819,-91.67969,41.699,-93.558
3952,11297,1259,2026-08-10,2026-08-20,2,28,260,41.529655,-90.48065,41.699,-93.558
3773,43338,2374,2026-08-10,2026-08-20,1,4,260,41.975787,-91.659795,41.699,-93.558
2512,43337,1111,2026-08-10,2026-08-20,3,38,260,41.642764,-91.530463,41.699,-93.558
2633,43337,1312,2026-08-10,2026-08-20,1,207,260,41.554101,-93.596754,41.699,-93.558
2633,43338,396,2026-08-10,2026-08-20,1,7,260,41.554101,-93.596754,41.699,-93.558
3952,26827,1285,2026-08-10,2026-08-20,1,39,85,41.529655,-90.48065,41.699,-93.558


## Verify
You should see 6 Silver tables: dim_store (20 rows), dim_item, dim_vendor,
dim_carrier, fact_sales (your full real history), fact_shipments (8 rows,
with vendor_number and dest_lat/dest_long correctly filled in from the joins).

If fact_shipments looks right, Step 7 is next: the Gold layer, which pulls
in the risk Bronze tables (OSRM/NWS/Open-Meteo) and reproduces your
gold_inventory_health / gold_shipments_open logic as real Delta tables.

In [0]:
display(spark.sql(f"SHOW TABLES IN {CATALOG}.silver"))

database,tableName,isTemporary
silver,dim_carrier,false
silver,dim_item,false
silver,dim_store,false
silver,dim_vendor,false
silver,fact_inventory_snapshot,false
silver,fact_sales,false
silver,fact_shipments,false


In [0]:
display(spark.sql("DESCRIBE pourcastai.silver.dim_store"))

col_name,data_type,comment
store_number,int,null
store_name,string,null
address,string,null
city,string,null
zip_code,string,null
latitude,double,null
longitude,double,null
population,int,null
rural_flag,int,null
county,string,null
